# 04 — 5-Fold Group Cross-Validation
**Goal:** Use `GroupKFold(n_splits=5)` with `groups=subjects` so every person appears
in the test fold exactly once. Inside each fold a small inner set of subjects is held
out for early-stopping validation only; the test fold is never touched during training.

Three 1D-CNN setups are compared:
- **Accel-only** (3 channels): ax, ay, az
- **Accel+Gyro** (6 channels): ax, ay, az, gx, gy, gz
- **Accel+Gyro+Mag** (8 channels): the 6 above + accel magnitude + gyro magnitude

All results are real numbers from actual training runs.

In [1]:
# ── Standard-library imports ──────────────────────────────────────────────────
import os       # file paths
import random   # Python RNG seeding
import warnings # suppress noisy deprecation messages
warnings.filterwarnings('ignore')

# ── Third-party imports ───────────────────────────────────────────────────────
import numpy as np    # array math
import pandas as pd   # results table

# GroupKFold splits data so every group (subject) appears in test exactly once
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score   # evaluation metrics

# ── TensorFlow / Keras imports ────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print('TensorFlow version:', tf.__version__)
print('Keras version     :', keras.__version__)
print('sklearn GroupKFold ready')

TensorFlow version: 2.19.0
Keras version     : 3.9.2
sklearn GroupKFold ready


In [2]:
# ── Fix all random seeds for reproducibility ──────────────────────────────────
SEED = 42   # master seed used everywhere

# Seed Python's built-in random module
random.seed(SEED)

# Seed NumPy's global RNG
np.random.seed(SEED)

# Seed TensorFlow (covers weight initialisers and dropout masks)
tf.random.set_seed(SEED)

# Deterministic Python hash ordering
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'All random seeds fixed to {SEED}')

All random seeds fixed to 42


In [3]:
# ── Path constants ────────────────────────────────────────────────────────────
DATA_PATH   = os.path.join('..', 'data', 'windows.npz')   # preprocessed windows
FIGURES_DIR = os.path.join('..', 'figures')               # output CSV directory

# Create figures directory if it does not exist
os.makedirs(FIGURES_DIR, exist_ok=True)

# Human-readable class names for the results table
CLASS_NAMES = ['Walking', 'Jogging', 'Stairs', 'Sitting', 'Standing']

# Training hyperparameters — same as notebook 02 for fair comparison
EPOCHS     = 60    # maximum epochs per run
BATCH_SIZE = 128   # mini-batch size

# Fraction of fold-train subjects to reserve for inner early-stopping validation
INNER_VAL_FRAC = 0.15

print('Data path  :', os.path.abspath(DATA_PATH))
print('Figures dir:', os.path.abspath(FIGURES_DIR))

Data path  : C:\Users\shiva\Downloads\HAR2\data\windows.npz
Figures dir: C:\Users\shiva\Downloads\HAR2\figures


In [4]:
# ── Load preprocessed windows ─────────────────────────────────────────────────
# windows.npz was created by notebook 01; contains merged + resampled sensor data
data     = np.load(DATA_PATH)
X        = data['X']         # shape (N, 100, 6) — ax ay az gx gy gz
y        = data['y']         # shape (N,)         — integer labels 0-4
subjects = data['subjects']  # shape (N,)         — subject ID per window

print(f'X shape        : {X.shape}')
print(f'y shape        : {y.shape}')
print(f'subjects shape : {subjects.shape}')
print(f'Unique subjects: {len(np.unique(subjects))}')
print(f'Unique labels  : {np.unique(y).tolist()}')

X shape        : (17685, 100, 6)
y shape        : (17685,)
subjects shape : (17685,)
Unique subjects: 51
Unique labels  : [0, 1, 2, 3, 4]


In [5]:
# ── Compute magnitude channels from RAW data ──────────────────────────────────
# Magnitudes are computed BEFORE normalisation so that the per-fold
# normalisation can compute proper mean/std for channels 6 and 7.
#
# accel_mag[i,t] = sqrt( ax[i,t]^2 + ay[i,t]^2 + az[i,t]^2 )
# gyro_mag[i,t]  = sqrt( gx[i,t]^2 + gy[i,t]^2 + gz[i,t]^2 )
#
# Both have shape (N, 100); we expand_dims to (N, 100, 1) for concatenation.

# Accelerometer magnitude along the last (channel) axis over channels 0,1,2
accel_mag = np.sqrt((X[:, :, :3] ** 2).sum(axis=2, keepdims=True))  # (N,100,1)

# Gyroscope magnitude over channels 3,4,5
gyro_mag  = np.sqrt((X[:, :, 3:] ** 2).sum(axis=2, keepdims=True))  # (N,100,1)

# Concatenate the original 6 channels with the 2 magnitude channels → 8 channels
X_8ch = np.concatenate([X, accel_mag, gyro_mag], axis=2)  # (N, 100, 8)

print(f'Original X shape : {X.shape}   (6 channels)')
print(f'X_8ch shape      : {X_8ch.shape}  (8 channels: 6 + accel_mag + gyro_mag)')

# Quick sanity check: magnitude of first window, first time step
expected = float(np.sqrt(X[0, 0, 0]**2 + X[0, 0, 1]**2 + X[0, 0, 2]**2))
got      = float(X_8ch[0, 0, 6])
print(f'\nSanity check accel_mag[0,0]: expected={expected:.4f}, got={got:.4f}')

Original X shape : (17685, 100, 6)   (6 channels)
X_8ch shape      : (17685, 100, 8)  (8 channels: 6 + accel_mag + gyro_mag)

Sanity check accel_mag[0,0]: expected=14.0645, got=14.0645


In [6]:
# ── Channel-selection helper ──────────────────────────────────────────────────
# Returns the correct data array for a given setup name.
# X_raw  : original (N, 100, 6) array
# X_8    : magnitude-augmented (N, 100, 8) array
# Both must be the same N windows (same ordering).

def get_channels(X_raw, X_8, setup):
    """Select the right channel subset for a given experiment setup.

    Parameters
    ----------
    X_raw : raw sensor array (N, 100, 6)
    X_8   : sensor + magnitude array (N, 100, 8)
    setup : one of 'accel', 'both', 'mag'

    Returns
    -------
    array and number of channels
    """
    if setup == 'accel':
        # Accel-only: slice first 3 channels (ax, ay, az)
        return X_raw[:, :, :3], 3
    elif setup == 'both':
        # All 6 original channels
        return X_raw, 6
    elif setup == 'mag':
        # All 8 channels including magnitude features
        return X_8, 8
    else:
        raise ValueError(f'Unknown setup: {setup}')

In [7]:
# ── Normalisation helper ───────────────────────────────────────────────────────
# Computes per-channel mean and std from the TRAINING fold ONLY, then applies
# the same statistics to validation and test folds.
# This is called fresh inside every fold so statistics never leak across folds.

def normalise(X_tr, X_v, X_te):
    """Z-score normalise using training-fold statistics only.

    Parameters
    ----------
    X_tr : training windows (N_tr, 100, n_ch)
    X_v  : inner-val windows (N_v,  100, n_ch)
    X_te : test-fold windows (N_te, 100, n_ch)

    Returns
    -------
    Normalised X_tr, X_v, X_te (same shapes)
    """
    # Flatten train windows along time dimension to compute channel statistics
    flat    = X_tr.reshape(-1, X_tr.shape[-1])  # (N_tr * 100, n_ch)
    mean    = flat.mean(axis=0)                   # (n_ch,)
    std     = flat.std(axis=0)                    # (n_ch,)

    # Guard: if any channel has zero variance, replace std with 1 to avoid NaN
    std[std == 0] = 1.0

    # Apply (x - mean) / std; broadcasting over (N, 100, n_ch)
    return (X_tr - mean) / std, (X_v - mean) / std, (X_te - mean) / std

In [8]:
# ── 1D-CNN builder (identical architecture to notebook 02) ────────────────────
# The input_shape argument lets the same function build 3-ch, 6-ch, or 8-ch models.

def build_cnn(input_shape, n_classes=5):
    """Build and compile a small 1D-CNN.

    Architecture (same as notebook 02):
      Conv1D(32, 5) → BatchNorm → ReLU
      Conv1D(64, 3) → BatchNorm → ReLU
      GlobalAveragePooling1D
      Dense(64, relu) → Dropout(0.3)
      Dense(5, softmax)
    """
    # Input layer: (100 timesteps, n_channels)
    inp = keras.Input(shape=input_shape, name='input')

    # First Conv block: 32 filters, kernel size 5, no bias (BatchNorm provides bias)
    x = layers.Conv1D(32, kernel_size=5, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Conv block: 64 filters, kernel size 3
    x = layers.Conv1D(64, kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Global average pooling: collapses the 100 time-steps to a 64-d vector
    x = layers.GlobalAveragePooling1D()(x)

    # Dense head with dropout for regularisation
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Softmax output over 5 activity classes
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    # Compile with Adam optimiser and sparse cross-entropy
    model = keras.Model(inputs=inp, outputs=out, name='CNN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Print parameter counts for each input size
for n_ch in [3, 6, 8]:
    m = build_cnn((100, n_ch))
    print(f'CNN ({n_ch}-ch) parameters: {m.count_params():,}')
    del m  # free memory

CNN (3-ch) parameters: 11,493
CNN (6-ch) parameters: 11,973
CNN (8-ch) parameters: 12,293


In [9]:
# ── 5-Fold GroupKFold Cross-Validation ───────────────────────────────────────
# GroupKFold guarantees that all windows belonging to a subject stay
# together — they all go to train OR all go to test in a given fold.
# This is the correct evaluation protocol for HAR because we want to
# know if the model generalises to UNSEEN PEOPLE, not just unseen windows.

N_FOLDS = 5   # number of cross-validation folds

# Three channel setups to compare
SETUPS = [
    ('accel',  'Accel-only (3ch)'),
    ('both',   'Accel+Gyro (6ch)'),
    ('mag',    'Accel+Gyro+Mag (8ch)'),
]

# Initialise GroupKFold splitter
gkf = GroupKFold(n_splits=N_FOLDS)

# Storage for per-fold results:
# fold_results[setup_key] = list of metric dicts, one per fold
fold_results = {key: [] for key, _ in SETUPS}

# ── Outer loop: iterate over the 5 folds ─────────────────────────────────────
# gkf.split returns (train_window_indices, test_window_indices) for each fold
for fold_idx, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups=subjects)):

    print(f'\n{"="*65}')
    print(f'FOLD {fold_idx + 1} / {N_FOLDS}')
    print(f'{"="*65}')

    # ── Identify test subjects for this fold ──────────────────────────────
    # All windows in test_idx belong to the same ~10 subjects
    test_subjs_fold  = np.unique(subjects[test_idx])

    # All windows in train_idx belong to the remaining ~41 subjects
    train_subjs_fold = np.unique(subjects[train_idx])

    print(f'Test  subjects ({len(test_subjs_fold)}): {sorted(test_subjs_fold.tolist())}')
    print(f'Train subjects ({len(train_subjs_fold)}): {sorted(train_subjs_fold.tolist())}')

    # ── Inner split: hold out ~15% of TRAIN subjects for early stopping ───
    # Use a per-fold seed so each fold's inner split is reproducible but
    # different from other folds (avoids always picking the same val subjects).
    inner_rng = np.random.default_rng(SEED + fold_idx)

    # Shuffle the fold's training subjects
    shuffled_train_subjs = inner_rng.permutation(train_subjs_fold)

    # Reserve the first 15% as inner-validation subjects
    n_inner_val = max(1, int(np.floor(INNER_VAL_FRAC * len(shuffled_train_subjs))))
    inner_val_subjs   = shuffled_train_subjs[:n_inner_val]
    inner_train_subjs = shuffled_train_subjs[n_inner_val:]

    print(f'Inner-val subjects ({len(inner_val_subjs)}): {sorted(inner_val_subjs.tolist())}')
    print(f'Inner-train subjects ({len(inner_train_subjs)}): {sorted(inner_train_subjs.tolist())}')

    # Build window-level boolean masks from subject-level sets
    inner_train_mask = np.isin(subjects, inner_train_subjs)  # windows for training
    inner_val_mask   = np.isin(subjects, inner_val_subjs)    # windows for ES val
    test_fold_mask   = np.isin(subjects, test_subjs_fold)    # windows for evaluation

    # Extract window arrays and labels for each split
    # Raw 6-channel arrays (used by accel and both setups)
    X_tr_raw = X[inner_train_mask];   y_tr = y[inner_train_mask]
    X_v_raw  = X[inner_val_mask];     y_v  = y[inner_val_mask]
    X_te_raw = X[test_fold_mask];     y_te = y[test_fold_mask]

    # 8-channel arrays (used by mag setup)
    X_tr_8 = X_8ch[inner_train_mask]
    X_v_8  = X_8ch[inner_val_mask]
    X_te_8 = X_8ch[test_fold_mask]

    print(f'Inner-train windows : {len(X_tr_raw):,}')
    print(f'Inner-val   windows : {len(X_v_raw):,}')
    print(f'Test-fold   windows : {len(X_te_raw):,}')

    # ── Inner loop: iterate over the 3 channel setups ─────────────────────
    for setup_key, setup_label in SETUPS:

        print(f'\n  --- Setup: {setup_label} ---')

        # Select the correct channel slice for train, val, and test
        Xtr_ch, n_ch = get_channels(X_tr_raw, X_tr_8, setup_key)
        Xv_ch,  _    = get_channels(X_v_raw,  X_v_8,  setup_key)
        Xte_ch, _    = get_channels(X_te_raw, X_te_8, setup_key)

        # Normalise: compute stats from training windows ONLY for this fold+setup
        Xtr_n, Xv_n, Xte_n = normalise(Xtr_ch, Xv_ch, Xte_ch)

        # Reset TF seed before each run for reproducible weight initialisation
        tf.random.set_seed(SEED + fold_idx * 10)

        # Build a fresh model with the correct input shape
        model = build_cnn((100, n_ch))
        print(f'  Parameters: {model.count_params():,}')

        # EarlyStopping: monitor inner-val loss, stop after 10 patient epochs,
        # and restore the weights from the epoch with the lowest val_loss
        early_stop = callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=0   # silent — we already print fold/setup headers
        )

        # Train on inner-train windows, validate on inner-val windows
        history = model.fit(
            Xtr_n, y_tr,
            validation_data=(Xv_n, y_v),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop],
            verbose=0   # silent training — print only final metrics
        )

        # Print how many epochs were actually run
        stopped_at = early_stop.stopped_epoch if early_stop.stopped_epoch > 0 else EPOCHS
        best_epoch = stopped_at - 10 if early_stop.stopped_epoch > 0 else EPOCHS
        print(f'  Stopped at epoch {stopped_at}, best epoch ~{best_epoch}')

        # ── Evaluate on the TEST FOLD (used only here, never for training) ─
        test_preds   = model.predict(Xte_n, verbose=0).argmax(axis=1)
        test_acc     = accuracy_score(y_te, test_preds)
        macro_f1     = f1_score(y_te, test_preds, average='macro')
        per_class_f1 = f1_score(y_te, test_preds, average=None, labels=[0,1,2,3,4])

        print(f'  Test accuracy : {test_acc:.4f}')
        print(f'  Test macro F1 : {macro_f1:.4f}')
        print(f'  Per-class F1  : {[round(v,3) for v in per_class_f1]}')

        # Store this fold's metrics for later aggregation
        fold_results[setup_key].append({
            'fold'     : fold_idx + 1,
            'acc'      : test_acc,
            'macro_f1' : macro_f1,
            'f1_per'   : per_class_f1.tolist(),   # list of 5 floats
        })

        # Free the model from GPU/CPU memory before building the next one
        del model
        keras.backend.clear_session()

print(f'\n{"="*65}')
print('All folds complete.')


FOLD 1 / 5
Test  subjects (10): [1611, 1615, 1617, 1627, 1634, 1635, 1643, 1647, 1648, 1649]
Train subjects (41): [1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1608, 1609, 1610, 1612, 1613, 1614, 1616, 1618, 1619, 1620, 1621, 1622, 1623, 1624, 1625, 1626, 1628, 1629, 1630, 1631, 1632, 1633, 1636, 1637, 1638, 1639, 1640, 1641, 1642, 1644, 1645, 1646, 1650]
Inner-val subjects (6): [1604, 1607, 1621, 1638, 1645, 1650]
Inner-train subjects (35): [1600, 1601, 1602, 1603, 1605, 1606, 1608, 1609, 1610, 1612, 1613, 1614, 1616, 1618, 1619, 1620, 1622, 1623, 1624, 1625, 1626, 1628, 1629, 1630, 1631, 1632, 1633, 1636, 1637, 1639, 1640, 1641, 1642, 1644, 1646]
Inner-train windows : 12,063
Inner-val   windows : 2,113
Test-fold   windows : 3,509

  --- Setup: Accel-only (3ch) ---


  Parameters: 11,493


  Stopped at epoch 20, best epoch ~10


  Test accuracy : 0.8247
  Test macro F1 : 0.8200
  Per-class F1  : [np.float64(0.812), np.float64(0.917), np.float64(0.793), np.float64(0.749), np.float64(0.829)]



  --- Setup: Accel+Gyro (6ch) ---


  Parameters: 11,973


... (33 outputs left out)

  Test accuracy : 0.8084
  Test macro F1 : 0.8089
  Per-class F1  : [np.float64(0.782), np.float64(0.826), np.float64(0.806), np.float64(0.805), np.float64(0.826)]

  --- Setup: Accel+Gyro+Mag (8ch) ---


  Parameters: 12,293


  Stopped at epoch 11, best epoch ~1


  Test accuracy : 0.7531
  Test macro F1 : 0.7245
  Per-class F1  : [np.float64(0.385), np.float64(0.869), np.float64(0.702), np.float64(0.812), np.float64(0.854)]

All folds complete.


In [10]:
# ── Print per-fold detail table ────────────────────────────────────────────────
# One row per (fold, setup) showing raw accuracy and macro F1 before aggregation

print('\n===== Per-Fold Results =====')
print(f'{"Setup":<25} {"Fold":>4} {"Accuracy":>9} {"Macro F1":>9}')
print('-' * 52)

for setup_key, setup_label in SETUPS:
    for res in fold_results[setup_key]:
        print(f'{setup_label:<25} {res["fold"]:>4} {res["acc"]:>9.4f} {res["macro_f1"]:>9.4f}')
    print()  # blank line between setups


===== Per-Fold Results =====
Setup                     Fold  Accuracy  Macro F1
----------------------------------------------------
Accel-only (3ch)             1    0.8247    0.8200
Accel-only (3ch)             2    0.6785    0.6842
Accel-only (3ch)             3    0.8058    0.8034
Accel-only (3ch)             4    0.8026    0.8067
Accel-only (3ch)             5    0.8019    0.8020

Accel+Gyro (6ch)             1    0.8156    0.8106
Accel+Gyro (6ch)             2    0.7324    0.7294
Accel+Gyro (6ch)             3    0.7695    0.7648
Accel+Gyro (6ch)             4    0.7905    0.7896
Accel+Gyro (6ch)             5    0.8084    0.8089

Accel+Gyro+Mag (8ch)         1    0.8472    0.8445
Accel+Gyro+Mag (8ch)         2    0.7883    0.7887
Accel+Gyro+Mag (8ch)         3    0.7747    0.7720
Accel+Gyro+Mag (8ch)         4    0.8212    0.8221
Accel+Gyro+Mag (8ch)         5    0.7531    0.7245



In [11]:
# ── Aggregate across folds: compute mean and std for each setup ────────────────
# For accuracy, macro F1, and each of the 5 per-class F1 scores,
# we compute the mean and std over the 5 fold values.

summary_rows = []   # one dict per setup for the final CSV

for setup_key, setup_label in SETUPS:
    results = fold_results[setup_key]   # list of 5 fold dicts

    # Stack accuracy values across folds into a 1-D array
    accs    = np.array([r['acc']      for r in results])   # shape (5,)
    mf1s    = np.array([r['macro_f1'] for r in results])   # shape (5,)

    # Stack per-class F1: each r['f1_per'] is a list of 5 → shape (5, 5)
    pcf1    = np.array([r['f1_per']   for r in results])   # shape (5 folds, 5 classes)

    # Build the summary row
    row = {
        'Setup'           : setup_label,
        'Mean Accuracy'   : round(accs.mean(), 4),
        'Std Accuracy'    : round(accs.std(),  4),
        'Mean Macro F1'   : round(mf1s.mean(), 4),
        'Std Macro F1'    : round(mf1s.std(),  4),
    }

    # Add per-class F1 mean (std omitted for brevity — visible in per-fold table)
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        row[f'Mean F1 {cls_name}'] = round(pcf1[:, cls_idx].mean(), 4)

    summary_rows.append(row)

# Convert to a DataFrame for display and saving
summary_df = pd.DataFrame(summary_rows)

# Display options: show all columns without wrapping
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

print('\n===== Cross-Validation Summary (mean ± std over 5 folds) =====')
print(summary_df.to_string(index=False))


===== Cross-Validation Summary (mean ± std over 5 folds) =====
               Setup  Mean Accuracy  Std Accuracy  Mean Macro F1  Std Macro F1  Mean F1 Walking  Mean F1 Jogging  Mean F1 Stairs  Mean F1 Sitting  Mean F1 Standing
    Accel-only (3ch)         0.7827        0.0528         0.7833        0.0499           0.6745           0.9153          0.6399           0.8443            0.8424
    Accel+Gyro (6ch)         0.7833        0.0300         0.7807        0.0305           0.6481           0.8892          0.7136           0.8334            0.8190
Accel+Gyro+Mag (8ch)         0.7969        0.0335         0.7904        0.0415           0.6113           0.9066          0.7472           0.8428            0.8439


In [12]:
# ── Save summary table to CSV ──────────────────────────────────────────────────
csv_path = os.path.join(FIGURES_DIR, 'cv_results.csv')

# Write the summary DataFrame without row indices
summary_df.to_csv(csv_path, index=False)
print(f'CV results saved to: {os.path.abspath(csv_path)}')

# Also print a cleaner two-column summary for easy reading
print('\n===== Quick Summary =====')
for _, row in summary_df.iterrows():
    print(f"{row['Setup']}")
    print(f"  Accuracy : {row['Mean Accuracy']:.4f} ± {row['Std Accuracy']:.4f}")
    print(f"  Macro F1 : {row['Mean Macro F1']:.4f} ± {row['Std Macro F1']:.4f}")
    # Print per-class means on one line
    pcf1_str = '  '.join(
        f"{cn[:4]}={row[f'Mean F1 {cn}']:.3f}" for cn in CLASS_NAMES
    )
    print(f"  Per-class F1: {pcf1_str}")
    print()

CV results saved to: C:\Users\shiva\Downloads\HAR2\figures\cv_results.csv

===== Quick Summary =====
Accel-only (3ch)
  Accuracy : 0.7827 ± 0.0528
  Macro F1 : 0.7833 ± 0.0499
  Per-class F1: Walk=0.674  Jogg=0.915  Stai=0.640  Sitt=0.844  Stan=0.842

Accel+Gyro (6ch)
  Accuracy : 0.7833 ± 0.0300
  Macro F1 : 0.7807 ± 0.0305
  Per-class F1: Walk=0.648  Jogg=0.889  Stai=0.714  Sitt=0.833  Stan=0.819

Accel+Gyro+Mag (8ch)
  Accuracy : 0.7969 ± 0.0335
  Macro F1 : 0.7904 ± 0.0415
  Per-class F1: Walk=0.611  Jogg=0.907  Stai=0.747  Sitt=0.843  Stan=0.844

